In [ ]:
import pandas as pd
import numpy as np
import re
import os

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
INPUT_STRUCTURED = "Information_Systems_structured.csv"
RAW_FILE = "Information_Systems_article_data.csv"

# If master schema CSV is one folder above your notebook folder:
MASTER_SCHEMA_PATH = "../All_Journals_CLEANED.csv"

OUTPUT_FINAL = "Information_Systems_final_for_powerBI.csv"
OUTPUT_UNI_REVIEW = "Information_Systems_university_review.csv"

In [ ]:
print("CWD:", os.getcwd())
print("Structured exists:", os.path.exists(INPUT_STRUCTURED))
print("Raw exists:", os.path.exists(RAW_FILE))
print("Master exists:", os.path.exists(MASTER_SCHEMA_PATH))

In [ ]:
df = pd.read_csv(INPUT_STRUCTURED)
print("Rows:", len(df))
df.head()

In [ ]:
required = [
    "URL", "Journal_Title", "Standardized_Title", "month_year", "Abstract", "Keywords",
    "Author_Name", "Standardized_Author", "Author_University", "Standardized_University",
    "Author_State", "Author_Country", "Author_Address"
]

for c in required:
    if c not in df.columns:
        df[c] = np.nan

In [ ]:
# Fill Standardized_Title from possible title columns
if "Standardized_Title" not in df.columns:
    df["Standardized_Title"] = np.nan

# Priority order of title sources
if "Article_Title" in df.columns:
    df["Standardized_Title"] = df["Standardized_Title"].fillna(df["Article_Title"])

elif "Title" in df.columns:
    df["Standardized_Title"] = df["Standardized_Title"].fillna(df["Title"])

In [ ]:
def clean_title(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip()
    s = re.sub(r"[☆★]+$", "", s)   # remove decorative markers
    s = re.sub(r"\s+", " ", s)     # normalize spaces
    return s

df["Standardized_Title"] = df["Standardized_Title"].apply(clean_title)

In [ ]:
def clean_spaces(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip()
    if not s:
        return np.nan
    s = re.sub(r"\s+", " ", s)
    return s

def clean_title(x):
    # remove trailing decorative markers, normalize whitespace
    s = clean_spaces(x)
    if pd.isna(s):
        return np.nan
    s = re.sub(r"[☆★]+$", "", s).strip()
    s = re.sub(r"\s+", " ", s)
    return s if s else np.nan

def clean_author(x):
    s = clean_spaces(x)
    if pd.isna(s):
        return np.nan
    return s.title()

def clean_university(x):
    return clean_spaces(x)

def extract_year_from_text(value):
    if pd.isna(value):
        return np.nan
    text = str(value)
    m = re.search(r"(19|20)\d{2}", text)
    return int(m.group(0)) if m else np.nan

In [ ]:
# URL normalize for joins
df["URL"] = df["URL"].astype(str).str.strip()

# Title standardization
df["Standardized_Title"] = df["Standardized_Title"].apply(clean_title)

# Author standardization (supports cases where Author_name exists)
if df["Author_Name"].isna().all() and "Author_name" in df.columns:
    df["Author_Name"] = df["Author_name"]

df["Standardized_Author"] = df["Standardized_Author"].fillna(df["Author_Name"])
df["Standardized_Author"] = df["Standardized_Author"].apply(clean_author)

# Keep Author_Name populated (Power BI friendliness)
df["Author_Name"] = df["Author_Name"].fillna(df["Standardized_Author"])
df["Author_Name"] = df["Author_Name"].apply(clean_spaces)

In [ ]:
if os.path.exists(RAW_FILE):
    raw = pd.read_csv(RAW_FILE)
    if "URL" in raw.columns:
        raw["URL"] = raw["URL"].astype(str).str.strip()

    # prefer Month_Year if available
    if "Month_Year" in raw.columns and "URL" in raw.columns:
        raw["year_extracted"] = raw["Month_Year"].apply(extract_year_from_text)
        year_map = raw.drop_duplicates("URL").set_index("URL")["year_extracted"]
        df["month_year"] = df["month_year"].fillna(df["URL"].map(year_map))

# clean month_year to numeric-ish where possible
df["month_year"] = df["month_year"].apply(extract_year_from_text).fillna(df["month_year"])

In [ ]:
US_STATES = {
    "AL","AK","AZ","AR","CA","CO","CT","DE","FL","GA","HI","ID","IL","IN","IA","KS","KY","LA",
    "ME","MD","MA","MI","MN","MS","MO","MT","NE","NV","NH","NJ","NM","NY","NC","ND","OH","OK",
    "OR","PA","RI","SC","SD","TN","TX","UT","VT","VA","WA","WV","WI","WY","DC"
}

def extract_country_from_address(addr):
    if pd.isna(addr):
        return np.nan
    parts = [p.strip() for p in str(addr).split(",") if p.strip()]
    return parts[-1] if parts else np.nan

def extract_state_from_address(addr):
    if pd.isna(addr):
        return np.nan
    parts = [p.strip() for p in str(addr).split(",") if p.strip()]
    for p in parts:
        if p.upper() in US_STATES:
            return p.upper()
    return np.nan

if "Author_Address" in df.columns:
    df["Author_Country"] = df["Author_Country"].fillna(df["Author_Address"].apply(extract_country_from_address))
    df["Author_State"] = df["Author_State"].fillna(df["Author_Address"].apply(extract_state_from_address))

df["Author_Country"] = df["Author_Country"].apply(clean_spaces)
df["Author_State"] = df["Author_State"].apply(clean_spaces)

In [ ]:
df["Author_University"] = df["Author_University"].apply(clean_university)

unique_unis = sorted([u for u in df["Author_University"].dropna().unique() if str(u).strip() != ""])
print("Unique universities:", len(unique_unis))

threshold = 0.85
mapping = {}

if len(unique_unis) > 1:
    vectorizer = TfidfVectorizer(analyzer="char", ngram_range=(2, 4), lowercase=True)
    X = vectorizer.fit_transform(unique_unis)
    sim = cosine_similarity(X)

    for i, uni_i in enumerate(unique_unis):
        scores = sim[i].copy()
        scores[i] = 0.0
        j = int(scores.argmax())
        best_score = float(scores[j])

        if best_score >= threshold:
            uni_j = unique_unis[j]
            canonical = uni_i if len(uni_i) >= len(uni_j) else uni_j
            duplicate = uni_j if canonical == uni_i else uni_i
            mapping[duplicate] = canonical

print("University mappings:", len(mapping))

df["University_Canonical"] = df["Author_University"].replace(mapping)

# Standardized_University should be canonical (fallback to Author_University)
df["Standardized_University"] = df["Standardized_University"].fillna(df["University_Canonical"])
df["Standardized_University"] = df["Standardized_University"].fillna(df["Author_University"])
df["Standardized_University"] = df["Standardized_University"].apply(clean_university)

In [ ]:
review_df = pd.DataFrame(
    [{"Duplicate_Name": k, "Canonical_Name": v} for k, v in mapping.items()]
).sort_values(["Canonical_Name", "Duplicate_Name"])

review_df.to_csv(OUTPUT_UNI_REVIEW, index=False)
print("Saved review:", OUTPUT_UNI_REVIEW)

In [ ]:
df = df.replace(["nan", "NaN", "None", "N/A", ""], np.nan)

In [ ]:
master = pd.read_csv(MASTER_SCHEMA_PATH)
master_cols = master.columns.tolist()

# Add missing columns
for c in master_cols:
    if c not in df.columns:
        df[c] = np.nan

# Keep only master columns in exact order
df_final = df[master_cols].copy()

print("Columns match master:", list(df_final.columns) == master_cols)
print("Final rows:", len(df_final))

df_final.to_csv(OUTPUT_FINAL, index=False)
print("Saved final Power BI file:", OUTPUT_FINAL)

In [ ]:
print("Null % (top 15):")
null_pct = (df_final.isna().mean() * 100).sort_values(ascending=False).head(15)
print(null_pct)

print("\nSample rows:")
df_final.head(3)